In [1]:
import numpy as np
import pandas as pd
import optuna 
import json
import ipynbname
import warnings
from sklearn.exceptions import ConvergenceWarning
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

In [2]:
data = pd.read_csv("../Data_Final_duy.csv")
data

,temp,wind,RH,P,co,no,no2,o3,so2,nh3,pm2_5_next1,pm10_next1,pm2_5_next2,pm10_next2,pm2_5_next3,pm10_next3,weather_pm2_5_next1,weather_pm10_next1,weather_pm2_5_next2,weather_pm10_next2,weather_pm2_5_next3,weather_pm10_next3,year,month,day,dayofweek,hour
0,0.358974,0.170213,0.808343,0.541667,1.000000,1.000000,0.880066,0.000000,1.000000,1.000000,279.26,321.64,280.09,316.42,267.37,299.51,33.0,34.0,29.0,29.0,25.0,26.0,2021,9,1,2,0
1,0.358974,0.085106,0.808343,0.458333,1.000000,1.000000,0.900709,0.000000,1.000000,1.000000,280.09,316.42,267.37,299.51,248.21,276.38,33.0,34.0,29.0,29.0,25.0,26.0,2021,9,1,2,1
2,0.358974,0.085106,0.808343,0.458333,1.000000,1.000000,0.920135,0.000000,1.000000,1.000000,267.37,299.51,248.21,276.38,228.91,254.06,33.0,34.0,29.0,29.0,25.0,26.0,2021,9,1,2,2
3,0.358974,0.085106,0.808343,0.375000,1.000000,1.000000,0.941999,0.000000,1.000000,1.000000,248.21,276.38,228.91,254.06,218.71,243.63,33.0,34.0,29.0,29.0,25.0,26.0,2021,9,1,2,3
4,0.358974,0.000000,0.808343,0.458333,1.000000,1.000000,0.945532,0.000000,1.000000,1.000000,228.91,254.06,218.71,243.63,212.32,238.87,33.0,34.0,29.0,29.0,25.0,26.0,2021,9,1,2,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35056,0.307692,0.723404,0.883413,0.125000,-0.174303,0.025317,-0.019374,0.676021,-0.241955,0.730877,1.43,2.06,1.41,2.23,1.63,2.68,17.0,17.0,18.0,18.0,21.0,21.0,2025,8,31,6,16
35057,0.358974,0.638298,0.883413,0.125000,-0.199120,0.011836,-0.011859,0.663788,-0.241955,0.724561,1.41,2.23,1.63,2.68,1.97,3.29,17.0,17.0,18.0,18.0,21.0,21.0,2025,8,31,6,17
35058,0.358974,0.382979,0.883413,0.208333,-0.196096,0.004804,0.014891,0.651006,-0.227246,0.766405,1.63,2.68,1.97,3.29,2.37,3.97,17.0,17.0,18.0,18.0,21.0,21.0,2025,8,31,6,18
35059,0.358974,0.255319,0.883413,0.208333,-0.182572,0.002414,0.036438,0.640237,-0.216657,0.794748,1.97,3.29,2.37,3.97,2.75,4.66,17.0,17.0,18.0,18.0,21.0,21.0,2025,8,31,6,19


# I. Create new features

In [3]:
def add_lag_feature(data, lag_list, cols_added, time_col="time"):
    df = data.copy()
    cols_valid = [c for c in cols_added if c in df.columns]
    lag_dict = {}

    for col in cols_valid:
        for lag in lag_list:
            new_col = f"{col}_lag_{lag}"
            lag_dict[new_col] = df[col].shift(lag)

    df = pd.concat([df, pd.DataFrame(lag_dict)], axis=1)
    
    return df

def add_rolling_mean(data, rolling_list, cols_added, time_col="time"):
    df = data.copy()
    cols_valid = [c for c in cols_added if c in df.columns]
    roll_dict = {}

    for col in cols_valid:
        for roll in rolling_list:
            new_col = f"{col}_rollmean_{roll}"
            roll_dict[new_col] = df[col].rolling(window=roll).mean()

    df = pd.concat([df, pd.DataFrame(roll_dict)], axis=1)

    return df

In [4]:
time_col = ["time"]
num_col = ["temp", "wind", "RH", "P", "co", "no", "no2", "o3", "so2", "nh3"]
cat_col = ['weather0', 'weather1', 'year', 'month', 'day', 'dayofweek', 'hour']
all_target_col = ["pm2_5_next1", "pm10_next1", "pm2_5_next2", "pm10_next2", "pm2_5_next3", "pm10_next3"]
target_col = ["pm2_5_next1", "pm2_5_next2", "pm2_5_next3"]

In [5]:
lag_list = list(range(1, 25))
rolling_list = [r*720 for r in range(1, 7)] 

data = add_lag_feature(data, lag_list, num_col+['weather0', 'weather1'])
data = add_rolling_mean(data, rolling_list, num_col)
data

,temp,wind,RH,P,co,no,no2,o3,so2,nh3,pm2_5_next1,pm10_next1,pm2_5_next2,pm10_next2,pm2_5_next3,pm10_next3,weather_pm2_5_next1,weather_pm10_next1,weather_pm2_5_next2,weather_pm10_next2,weather_pm2_5_next3,weather_pm10_next3,year,month,day,dayofweek,hour,temp_lag_1,temp_lag_2,temp_lag_3,temp_lag_4,temp_lag_5,temp_lag_6,temp_lag_7,temp_lag_8,temp_lag_9,temp_lag_10,temp_lag_11,temp_lag_12,temp_lag_13,temp_lag_14,temp_lag_15,temp_lag_16,temp_lag_17,temp_lag_18,temp_lag_19,temp_lag_20,temp_lag_21,temp_lag_22,temp_lag_23,temp_lag_24,wind_lag_1,wind_lag_2,wind_lag_3,wind_lag_4,wind_lag_5,wind_lag_6,wind_lag_7,wind_lag_8,wind_lag_9,wind_lag_10,wind_lag_11,wind_lag_12,wind_lag_13,wind_lag_14,wind_lag_15,wind_lag_16,wind_lag_17,wind_lag_18,wind_lag_19,wind_lag_20,wind_lag_21,wind_lag_22,wind_lag_23,wind_lag_24,RH_lag_1,RH_lag_2,RH_lag_3,RH_lag_4,RH_lag_5,RH_lag_6,RH_lag_7,RH_lag_8,RH_lag_9,RH_lag_10,RH_lag_11,RH_lag_12,RH_lag_13,RH_lag_14,RH_lag_15,RH_lag_16,RH_lag_17,RH_lag_18,RH_lag_19,RH_lag_20,RH_lag_21,RH_lag_22,RH_lag_23,RH_lag_24,P_lag_1,P_lag_2,P_lag_3,P_lag_4,P_lag_5,P_lag_6,P_lag_7,P_lag_8,P_lag_9,P_lag_10,P_lag_11,P_lag_12,P_lag_13,P_lag_14,P_lag_15,P_lag_16,P_lag_17,P_lag_18,P_lag_19,P_lag_20,P_lag_21,P_lag_22,P_lag_23,P_lag_24,co_lag_1,co_lag_2,co_lag_3,co_lag_4,co_lag_5,co_lag_6,co_lag_7,co_lag_8,co_lag_9,co_lag_10,co_lag_11,co_lag_12,co_lag_13,co_lag_14,co_lag_15,co_lag_16,co_lag_17,co_lag_18,co_lag_19,co_lag_20,co_lag_21,co_lag_22,co_lag_23,co_lag_24,no_lag_1,no_lag_2,no_lag_3,no_lag_4,no_lag_5,no_lag_6,no_lag_7,no_lag_8,no_lag_9,no_lag_10,no_lag_11,no_lag_12,no_lag_13,no_lag_14,no_lag_15,no_lag_16,no_lag_17,no_lag_18,no_lag_19,no_lag_20,no_lag_21,no_lag_22,no_lag_23,no_lag_24,no2_lag_1,no2_lag_2,no2_lag_3,no2_lag_4,no2_lag_5,no2_lag_6,no2_lag_7,no2_lag_8,no2_lag_9,no2_lag_10,no2_lag_11,no2_lag_12,no2_lag_13,no2_lag_14,no2_lag_15,no2_lag_16,no2_lag_17,no2_lag_18,no2_lag_19,no2_lag_20,no2_lag_21,no2_lag_22,no2_lag_23,no2_lag_24,o3_lag_1,o3_lag_2,o3_lag_3,o3_lag_4,o3_lag_5,o3_lag_6,o3_lag_7,o3_lag_8,o3_lag_9,o3_lag_10,o3_lag_11,o3_lag_12,o3_lag_13,o3_lag_14,o3_lag_15,o3_lag_16,o3_lag_17,o3_lag_18,o3_lag_19,o3_lag_20,o3_lag_21,o3_lag_22,o3_lag_23,o3_lag_24,so2_lag_1,so2_lag_2,so2_lag_3,so2_lag_4,so2_lag_5,so2_lag_6,so2_lag_7,so2_lag_8,so2_lag_9,so2_lag_10,so2_lag_11,so2_lag_12,so2_lag_13,so2_lag_14,so2_lag_15,so2_lag_16,so2_lag_17,so2_lag_18,so2_lag_19,so2_lag_20,so2_lag_21,so2_lag_22,so2_lag_23,so2_lag_24,nh3_lag_1,nh3_lag_2,nh3_lag_3,nh3_lag_4,nh3_lag_5,nh3_lag_6,nh3_lag_7,nh3_lag_8,nh3_lag_9,nh3_lag_10,nh3_lag_11,nh3_lag_12,nh3_lag_13,nh3_lag_14,nh3_lag_15,nh3_lag_16,nh3_lag_17,nh3_lag_18,nh3_lag_19,nh3_lag_20,nh3_lag_21,nh3_lag_22,nh3_lag_23,nh3_lag_24,temp_rollmean_720,temp_rollmean_1440,temp_rollmean_2160,temp_rollmean_2880,temp_rollmean_3600,temp_rollmean_4320,wind_rollmean_720,wind_rollmean_1440,wind_rollmean_2160,wind_rollmean_2880,wind_rollmean_3600,wind_rollmean_4320,RH_rollmean_720,RH_rollmean_1440,RH_rollmean_2160,RH_rollmean_2880,RH_rollmean_3600,RH_rollmean_4320,P_rollmean_720,P_rollmean_1440,P_rollmean_2160,P_rollmean_2880,P_rollmean_3600,P_rollmean_4320,co_rollmean_720,co_rollmean_1440,co_rollmean_2160,co_rollmean_2880,co_rollmean_3600,co_rollmean_4320,no_rollmean_720,no_rollmean_1440,no_rollmean_2160,no_rollmean_2880,no_rollmean_3600,no_rollmean_4320,no2_rollmean_720,no2_rollmean_1440,no2_rollmean_2160,no2_rollmean_2880,no2_rollmean_3600,no2_rollmean_4320,o3_rollmean_720,o3_rollmean_1440,o3_rollmean_2160,o3_rollmean_2880,o3_rollmean_3600,o3_rollmean_4320,so2_rollmean_720,so2_rollmean_1440,so2_rollmean_2160,so2_rollmean_2880,so2_rollmean_3600,so2_rollmean_4320,nh3_rollmean_720,nh3_rollmean_1440,nh3_rollmean_2160,nh3_rollmean_2880,nh3_rollmean_3600,nh3_rollmean_4320
0,0.358974,0.170213,0.808343,0.541667,1.000000,1.000000,0.880066,0.000000,1.000000,1.000000,279.26,321.64,280.09,316.42,267.37,299.51,33.0,34.0,29.0,29.0,25.0,26.0,2021,9,1,2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

In [6]:
train = data[0:26304].reset_index(drop=True)
val   = data[-(4344+4413):-4413].reset_index(drop=True)
test  = data[-4413:].reset_index(drop=True)

In [7]:
train = train.dropna().reset_index(drop=True)
train

,temp,wind,RH,P,co,no,no2,o3,so2,nh3,pm2_5_next1,pm10_next1,pm2_5_next2,pm10_next2,pm2_5_next3,pm10_next3,weather_pm2_5_next1,weather_pm10_next1,weather_pm2_5_next2,weather_pm10_next2,weather_pm2_5_next3,weather_pm10_next3,year,month,day,dayofweek,hour,temp_lag_1,temp_lag_2,temp_lag_3,temp_lag_4,temp_lag_5,temp_lag_6,temp_lag_7,temp_lag_8,temp_lag_9,temp_lag_10,temp_lag_11,temp_lag_12,temp_lag_13,temp_lag_14,temp_lag_15,temp_lag_16,temp_lag_17,temp_lag_18,temp_lag_19,temp_lag_20,temp_lag_21,temp_lag_22,temp_lag_23,temp_lag_24,wind_lag_1,wind_lag_2,wind_lag_3,wind_lag_4,wind_lag_5,wind_lag_6,wind_lag_7,wind_lag_8,wind_lag_9,wind_lag_10,wind_lag_11,wind_lag_12,wind_lag_13,wind_lag_14,wind_lag_15,wind_lag_16,wind_lag_17,wind_lag_18,wind_lag_19,wind_lag_20,wind_lag_21,wind_lag_22,wind_lag_23,wind_lag_24,RH_lag_1,RH_lag_2,RH_lag_3,RH_lag_4,RH_lag_5,RH_lag_6,RH_lag_7,RH_lag_8,RH_lag_9,RH_lag_10,RH_lag_11,RH_lag_12,RH_lag_13,RH_lag_14,RH_lag_15,RH_lag_16,RH_lag_17,RH_lag_18,RH_lag_19,RH_lag_20,RH_lag_21,RH_lag_22,RH_lag_23,RH_lag_24,P_lag_1,P_lag_2,P_lag_3,P_lag_4,P_lag_5,P_lag_6,P_lag_7,P_lag_8,P_lag_9,P_lag_10,P_lag_11,P_lag_12,P_lag_13,P_lag_14,P_lag_15,P_lag_16,P_lag_17,P_lag_18,P_lag_19,P_lag_20,P_lag_21,P_lag_22,P_lag_23,P_lag_24,co_lag_1,co_lag_2,co_lag_3,co_lag_4,co_lag_5,co_lag_6,co_lag_7,co_lag_8,co_lag_9,co_lag_10,co_lag_11,co_lag_12,co_lag_13,co_lag_14,co_lag_15,co_lag_16,co_lag_17,co_lag_18,co_lag_19,co_lag_20,co_lag_21,co_lag_22,co_lag_23,co_lag_24,no_lag_1,no_lag_2,no_lag_3,no_lag_4,no_lag_5,no_lag_6,no_lag_7,no_lag_8,no_lag_9,no_lag_10,no_lag_11,no_lag_12,no_lag_13,no_lag_14,no_lag_15,no_lag_16,no_lag_17,no_lag_18,no_lag_19,no_lag_20,no_lag_21,no_lag_22,no_lag_23,no_lag_24,no2_lag_1,no2_lag_2,no2_lag_3,no2_lag_4,no2_lag_5,no2_lag_6,no2_lag_7,no2_lag_8,no2_lag_9,no2_lag_10,no2_lag_11,no2_lag_12,no2_lag_13,no2_lag_14,no2_lag_15,no2_lag_16,no2_lag_17,no2_lag_18,no2_lag_19,no2_lag_20,no2_lag_21,no2_lag_22,no2_lag_23,no2_lag_24,o3_lag_1,o3_lag_2,o3_lag_3,o3_lag_4,o3_lag_5,o3_lag_6,o3_lag_7,o3_lag_8,o3_lag_9,o3_lag_10,o3_lag_11,o3_lag_12,o3_lag_13,o3_lag_14,o3_lag_15,o3_lag_16,o3_lag_17,o3_lag_18,o3_lag_19,o3_lag_20,o3_lag_21,o3_lag_22,o3_lag_23,o3_lag_24,so2_lag_1,so2_lag_2,so2_lag_3,so2_lag_4,so2_lag_5,so2_lag_6,so2_lag_7,so2_lag_8,so2_lag_9,so2_lag_10,so2_lag_11,so2_lag_12,so2_lag_13,so2_lag_14,so2_lag_15,so2_lag_16,so2_lag_17,so2_lag_18,so2_lag_19,so2_lag_20,so2_lag_21,so2_lag_22,so2_lag_23,so2_lag_24,nh3_lag_1,nh3_lag_2,nh3_lag_3,nh3_lag_4,nh3_lag_5,nh3_lag_6,nh3_lag_7,nh3_lag_8,nh3_lag_9,nh3_lag_10,nh3_lag_11,nh3_lag_12,nh3_lag_13,nh3_lag_14,nh3_lag_15,nh3_lag_16,nh3_lag_17,nh3_lag_18,nh3_lag_19,nh3_lag_20,nh3_lag_21,nh3_lag_22,nh3_lag_23,nh3_lag_24,temp_rollmean_720,temp_rollmean_1440,temp_rollmean_2160,temp_rollmean_2880,temp_rollmean_3600,temp_rollmean_4320,wind_rollmean_720,wind_rollmean_1440,wind_rollmean_2160,wind_rollmean_2880,wind_rollmean_3600,wind_rollmean_4320,RH_rollmean_720,RH_rollmean_1440,RH_rollmean_2160,RH_rollmean_2880,RH_rollmean_3600,RH_rollmean_4320,P_rollmean_720,P_rollmean_1440,P_rollmean_2160,P_rollmean_2880,P_rollmean_3600,P_rollmean_4320,co_rollmean_720,co_rollmean_1440,co_rollmean_2160,co_rollmean_2880,co_rollmean_3600,co_rollmean_4320,no_rollmean_720,no_rollmean_1440,no_rollmean_2160,no_rollmean_2880,no_rollmean_3600,no_rollmean_4320,no2_rollmean_720,no2_rollmean_1440,no2_rollmean_2160,no2_rollmean_2880,no2_rollmean_3600,no2_rollmean_4320,o3_rollmean_720,o3_rollmean_1440,o3_rollmean_2160,o3_rollmean_2880,o3_rollmean_3600,o3_rollmean_4320,so2_rollmean_720,so2_rollmean_1440,so2_rollmean_2160,so2_rollmean_2880,so2_rollmean_3600,so2_rollmean_4320,nh3_rollmean_720,nh3_rollmean_1440,nh3_rollmean_2160,nh3_rollmean_2880,nh3_rollmean_3600,nh3_rollmean_4320
0,0.461538,0.638298,0.357926,0.791667,0.234853,0.000000,0.366986,0.846914,0.188052,0.638729,8.78,17.57,8.76,17.09,9.21,18.10,33.0,34.0,29.0,29.0,25.0,26.0,2022,2,27,6,23,0.512821,0.512821,0.564103,0.564103,0.461538,0.512821,0.717949,0.820513,0.820

In [8]:
X_train, y_train = train.drop(columns=all_target_col), train[target_col]
X_val, y_val     = val.drop(columns=all_target_col), val[target_col]
X_test, y_test   = test.drop(columns=all_target_col), test[target_col]

# II. Linear Regression

In [9]:
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import ElasticNet
import pandas as pd
import numpy as np

def objective_LR(trial, X_train, y_train, X_val, y_val, X_test, y_test):
    # Hyperparameters
    alpha = trial.suggest_float("alpha", 0.001, 0.2, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)
    fit_intercept = trial.suggest_categorical("fit_intercept", [True, False])
    
    base_model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, fit_intercept=fit_intercept, max_iter=1000)
    model = MultiOutputRegressor(base_model)
    
    # Fit model
    model.fit(X_train, y_train)
    
    # Predict
    y_pred_train = pd.DataFrame(model.predict(X_train), columns=y_train.columns, index=y_train.index)
    y_pred_val   = pd.DataFrame(model.predict(X_val), columns=y_val.columns, index=y_val.index)
    y_pred_test  = pd.DataFrame(model.predict(X_test), columns=y_test.columns, index=y_test.index)

    
    # Compute RMSE từng target
    rmse_train = np.sqrt(np.mean((y_train - y_pred_train)**2, axis=0))
    rmse_val   = np.sqrt(np.mean((y_val - y_pred_val)**2, axis=0))
    rmse_test  = np.sqrt(np.mean((y_test - y_pred_test)**2, axis=0))

    # In RMSE từng target
    print(f"Trial {trial.number} RMSE Train/Val/Test per target:")
    for target, r_train, r_val, r_test in zip(y_train.columns, rmse_train, rmse_val, rmse_test):
        print(f" {target}: Train={r_train:.3f}, Val={r_val:.3f}, Test={r_test:.3f}")
    
    # Trung bình tất cả target
    rmse_val_mean = rmse_val.mean()
    return rmse_val_mean

In [10]:
warnings.filterwarnings("ignore", category=ConvergenceWarning)
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objective_LR(trial, X_train, y_train, 
                                          X_val, y_val, 
                                          X_test, y_test), 
               n_trials=15)

print("Best trial params:", study.best_trial.params)
print("Best RMSE:", study.best_trial.value)

[I 2025-12-27 02:12:34,320] A new study created in memory with name: no-name-701319c6-1fba-42b9-ad50-d11931c9e6f1
[I 2025-12-27 02:14:07,605] Trial 0 finished with value: 48.23293071113354 and parameters: {'alpha': 0.042942167796022834, 'l1_ratio': 0.15655948218575644, 'fit_intercept': True}. Best is trial 0 with value: 48.23293071113354.


Trial 0 RMSE Train/Val/Test per target:
 pm2_5_next1: Train=37.066, Val=44.950, Test=66.173
 pm2_5_next2: Train=38.991, Val=48.138, Test=62.955
 pm2_5_next3: Train=41.085, Val=51.611, Test=59.705


[I 2025-12-27 02:15:33,116] Trial 1 finished with value: 50.095373304671824 and parameters: {'alpha': 0.14071596542285914, 'l1_ratio': 0.24213783835651792, 'fit_intercept': False}. Best is trial 0 with value: 48.23293071113354.


Trial 1 RMSE Train/Val/Test per target:
 pm2_5_next1: Train=39.345, Val=47.257, Test=47.476
 pm2_5_next2: Train=41.108, Val=50.050, Test=45.055
 pm2_5_next3: Train=42.949, Val=52.979, Test=42.683


[I 2025-12-27 02:16:16,122] Trial 2 finished with value: 49.49077117252045 and parameters: {'alpha': 0.16071337541156994, 'l1_ratio': 0.6924137390314934, 'fit_intercept': True}. Best is trial 0 with value: 48.23293071113354.


Trial 2 RMSE Train/Val/Test per target:
 pm2_5_next1: Train=38.114, Val=46.402, Test=57.363
 pm2_5_next2: Train=39.961, Val=49.443, Test=55.357
 pm2_5_next3: Train=41.939, Val=52.627, Test=53.348


... (20 outputs left out)

[I 2025-12-27 02:31:36,550] Trial 13 finished with value: 43.01494665462538 and parameters: {'alpha': 0.0010349575523809428, 'l1_ratio': 0.9896790515470627, 'fit_intercept': True}. Best is trial 10 with value: 42.85591953334076.


Trial 13 RMSE Train/Val/Test per target:
 pm2_5_next1: Train=33.414, Val=39.838, Test=106.763
 pm2_5_next2: Train=35.331, Val=42.738, Test=93.572
 pm2_5_next3: Train=37.719, Val=46.469, Test=75.175


[I 2025-12-27 02:33:01,250] Trial 14 finished with value: 44.10209634132678 and parameters: {'alpha': 0.007304738360771309, 'l1_ratio': 0.7936966269335559, 'fit_intercept': True}. Best is trial 10 with value: 42.85591953334076.


Trial 14 RMSE Train/Val/Test per target:
 pm2_5_next1: Train=34.442, Val=40.499, Test=85.695
 pm2_5_next2: Train=36.413, Val=43.897, Test=80.969
 pm2_5_next3: Train=38.756, Val=47.910, Test=74.847
Best trial params: {'alpha': 0.0010192351864378793, 'l1_ratio': 0.9816839415336018, 'fit_intercept': True}
Best RMSE: 42.85591953334076


In [11]:
nb_path = ipynbname.path().stem  
save_path = f"Best parameters for {nb_path}.json"
best_params = study.best_trial.params
with open(save_path, "w") as f:
    json.dump(best_params, f, indent=4)

print(f"Saved best params to {save_path}")

Saved best params to Best parameters for PM2.5 - Linear Regression.json


# III. Visualization

In [12]:
from sklearn.linear_model import ElasticNet
from sklearn.multioutput import MultiOutputRegressor
import json

with open(r".\Best parameters for PM2.5 - Linear Regression.json", "r") as f: params = json.load(f)

base_model = ElasticNet(
    alpha=params['alpha'],
    l1_ratio=params['l1_ratio'],
    fit_intercept=params['fit_intercept'],
    max_iter=10000,  
    random_state=42
)
model = MultiOutputRegressor(base_model)
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_val   = model.predict(X_val)

y_pred_train_df = pd.DataFrame(y_pred_train, columns=y_train.columns).clip(lower=0)
y_pred_val_df   = pd.DataFrame(y_pred_val, columns=y_train.columns).clip(lower=0)

In [13]:
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr
import numpy as np

for target in target_col:
    # Train metrics
    y_true_train = y_train[target]
    y_pred_train = y_pred_train_df[target]
    
    rmse_train = np.sqrt(mean_squared_error(y_true_train, y_pred_train))
    r2_train = r2_score(y_true_train, y_pred_train)
    pearson_train, _ = pearsonr(y_true_train, y_pred_train)
    
    print(f"Train - Target: {target}")
    print(f"RMSE: {rmse_train:.4f}, R2: {r2_train:.4f}, Pearson: {pearson_train:.4f}\n")
    
    # Validation metrics
    y_true_val = y_val[target]
    y_pred_val = y_pred_val_df[target]
    
    rmse_val = np.sqrt(mean_squared_error(y_true_val, y_pred_val))
    r2_val = r2_score(y_true_val, y_pred_val)
    pearson_val, _ = pearsonr(y_true_val, y_pred_val)
    
    print(f"Validation - Target: {target}")
    print(f"RMSE: {rmse_val:.4f}, R2: {r2_val:.4f}, Pearson: {pearson_val:.4f}\n")

Train - Target: pm2_5_next1
RMSE: 32.6179, R2: 0.7083, Pearson: 0.8434

Validation - Target: pm2_5_next1
RMSE: 39.5083, R2: 0.6583, Pearson: 0.8284

Train - Target: pm2_5_next2
RMSE: 34.6331, R2: 0.6712, Pearson: 0.8209

Validation - Target: pm2_5_next2
RMSE: 42.5746, R2: 0.6030, Pearson: 0.7924

Train - Target: pm2_5_next3
RMSE: 37.1736, R2: 0.6212, Pearson: 0.7895

Validation - Target: pm2_5_next3
RMSE: 46.5294, R2: 0.5254, Pearson: 0.7395

